# RDF Metadata Harvesting

`harvest_publication_metadata_from_rdf` reads publication metadata from local RDF data and returns a review. Every value keeps the RDF statements behind it, and missing or conflicting information stays visible until you resolve it. Metadata only comes out of the review once it is complete.

This notebook walks through a review, from loading a file to building the payloads for Dataportal and Zenodo. It needs no network access or token, and nothing is published.

Run it from a clone of the repository. It reads the two PMD tensile-test reference files in `tests/resources/rdf_metadata/corpus/`, which a Binder session doesn't include. Those files aren't covered by praeco's BSD 3-Clause licence; see the [README next to them](../tests/resources/rdf_metadata/corpus/README.md).

In [1]:
from datetime import date
from pathlib import Path

from praeco import Person, harvest_publication_metadata_from_rdf
from praeco.exceptions import ValidationError
from praeco.rdf_metadata import IncompleteHarvestError
from praeco.services.dataportal import DataportalMetadata
from praeco.services.zenodo import ZenodoMetadata

CORPUS = Path("../tests/resources/rdf_metadata/corpus")

## Load a file

A `Path` is read as a local Turtle file. An `rdflib.Graph` is accepted too, and a `str` or `bytes` value is always Turtle content, never a path or URL.

This file is a tensile-test dataset that uses PMDco 3 terms. The source record describes what was read, without the local path:

In [2]:
harvest = harvest_publication_metadata_from_rdf(CORPUS / "S355_data_tto_pmdco3.ttl")
harvest.source

SourceInfo(kind='path', serialization='turtle', triple_count=6876, size_bytes=463935, sha256='b67d95710ab607bc0c93ec3df83f072a20bc07ac58adc020a2c1b666ddc8f25d')

You can check that you are reviewing the file you expect:

In [3]:
harvest.source.matches_checksum(
    "b67d95710ab607bc0c93ec3df83f072a20bc07ac58adc020a2c1b666ddc8f25d"
)

True

A harvest concerns one publication subject, and nothing is selected yet. The file describes many RDF resources:

In [4]:
print(len(harvest.subjects), "subjects")
for diagnostic in harvest.diagnostics:
    print(f"{diagnostic.code}: {diagnostic.message}")

1808 subjects
subject_required: Select a publication subject before reviewing metadata.


## Select the publication subject

`subject_candidates` lists the resources that could be the publication, with the statements that qualify them. `select_subject_automatically()` selects a subject only when exactly one candidate is eligible. Otherwise the harvest stays unresolved and you choose with `select_subject`, as shown at the end of this notebook.

In [5]:
for candidate in harvest.subject_candidates:
    print(candidate.subject.term, "eligible:", candidate.eligible)
    for evidence in candidate.evidence:
        print(f"  {evidence.rule}: {evidence.triples[0]}")

https://w3id.org/pmd/demodata/tensiletest_S355/ eligible: True
  publication_type: ('<https://w3id.org/pmd/demodata/tensiletest_S355/>', '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>', '<http://www.w3.org/2002/07/owl#Ontology>')
  publication_title: ('<https://w3id.org/pmd/demodata/tensiletest_S355/>', '<http://purl.org/dc/elements/1.1/title>', '"TTO/PMDco 3.0 A-Box mapping example of S355 steel tensile test data"^^<http://www.w3.org/2001/XMLSchema#string>')


Review operations return a new harvest and leave the previous one unchanged, so each step below gets its own name.

In [6]:
selected = harvest.select_subject_automatically()
print(selected.subject.term)
print(selected.subject_origin)

https://w3id.org/pmd/demodata/tensiletest_S355/
automatic


## Inspect the review

Each field reports a status: `missing`, `unresolved`, `resolved`, or `excluded`. In this file, only the title comes directly from the RDF.

In [7]:
fields = ("title", "description", "creators", "publication_date", "keywords", "license")
for name in fields:
    field = getattr(selected, name)
    print(f"{name}: {field.status}, value={field.value!r}")

title: resolved, value='TTO/PMDco 3.0 A-Box mapping example of S355 steel tensile test data'
description: missing, value=None
creators: missing, value=()
publication_date: missing, value=None
keywords: missing, value=None
license: missing, value=None


Conversion fails while required information is missing, and says why:

In [8]:
try:
    selected.to_publication_metadata()
except IncompleteHarvestError as error:
    print(error)
else:
    raise AssertionError("conversion should fail while the review is incomplete")

description requires review.; creators requires a non-empty reviewed list.


## Review the suggestions

A suggestion is a value derived from the RDF. It changes nothing until you accept it, and an unaccepted suggestion doesn't block conversion.

This file matches praeco's rules for PMD tensile-test data, so it gets a description and keywords. Its `dct:created` statement also gives a publication-date suggestion, because a creation date isn't automatically a publication date.

In [9]:
description = selected.description.suggestions[0]
keywords = selected.keywords.suggestions[0]
publication_date = selected.publication_date.suggestions[0]

print(description.value)
print(keywords.value)
print(publication_date.value)

assert description.value == (
    "Data from 10 tensile tests. Material: S355. Test standard: DIN EN ISO 6892-1."
)
assert keywords.value == ("S355", "tensile test")

Data from 10 tensile tests. Material: S355. Test standard: DIN EN ISO 6892-1.
('S355', 'tensile test')
2026-04-16


Each suggestion keeps the RDF statements it was derived from, grouped by the rule that found them:

In [10]:
for evidence in description.evidence:
    print(f"{evidence.rule}: {len(evidence.triples)} triples")
print(description.evidence[0].triples[0])

pmdco3_tensile_test: 20 triples
pmdco3_material: 40 triples
pmdco3_test_standard: 50 triples
('<https://w3id.org/pmd/demodata/tensiletest_S355/pmdao-tto-tt-S355-10_process>', '<http://purl.obolibrary.org/obo/OBI_0000293>', '<https://w3id.org/pmd/demodata/tensiletest_S355/testpiece/Zd3>')


Accept what you have checked. Treating the creation date as the publication date is a reviewer's decision, made here for the demonstration.

In [11]:
accepted = (
    selected.accept_suggestion(description)
    .accept_suggestion(keywords)
    .accept_suggestion(publication_date)
)
print(accepted.description.value)
print(accepted.description.origin)

assert selected.description.value is None  # the previous harvest is unchanged

Data from 10 tensile tests. Material: S355. Test standard: DIN EN ISO 6892-1.
acceptance


## Complete the review and convert

The file names no creators, and praeco doesn't invent any. `with_overrides` takes values you have reviewed; the name below is a placeholder for your real list of `Person` or `Organization` creators.

In [12]:
reviewed = accepted.with_overrides(creators=(Person(name="Jane Doe"),))
metadata = reviewed.to_publication_metadata()

assert metadata.description == description.value
assert metadata.keywords == keywords.value
assert metadata.publication_date == date(2026, 4, 16)
assert metadata.creators == (Person(name="Jane Doe"),)

metadata

PublicationMetadata(title='TTO/PMDco 3.0 A-Box mapping example of S355 steel tensile test data', description='Data from 10 tensile tests. Material: S355. Test standard: DIN EN ISO 6892-1.', creators=(Person(family_name=None, given_names=None, name='Jane Doe', affiliation=None, orcid=None, gnd=None),), publication_date=datetime.date(2026, 4, 16), contributors=(), keywords=('S355', 'tensile test'), license=None, doi=None, version=None, language=None, related_identifiers=())

The review records where each value came from:

In [13]:
for name in ("title", "description", "creators", "publication_date", "keywords"):
    print(f"{name}: {getattr(reviewed, name).origin}")

title: automatic
description: acceptance
creators: override
publication_date: acceptance
keywords: acceptance


## Build service payloads

The neutral `PublicationMetadata` goes through the existing adapters, which keep their own validation and defaults. `to_payload()` only builds the request body; nothing is sent. `DataportalMetadata` doesn't require a license:

In [14]:
dataportal_payload = DataportalMetadata(metadata=metadata).to_payload()

assert dataportal_payload["notes"] == description.value
assert "license_id" not in dataportal_payload

dataportal_payload

{'name': 'tto-pmdco-3-0-a-box-mapping-example-of-s355-steel-tensile-test-data',
 'title': 'TTO/PMDco 3.0 A-Box mapping example of S355 steel tensile test data',
 'notes': 'Data from 10 tensile tests. Material: S355. Test standard: DIN EN ISO 6892-1.',
 'tags': [{'name': 'S355'}, {'name': 'tensile test'}],
 'creator': [{'name': 'Jane Doe', 'type': 'Person'}],
 'issued': '2026-04-16',
 'extras': [{'key': 'publication_date', 'value': '2026-04-16'},
  {'key': 'creators', 'value': '[{"name":"Jane Doe"}]'}]}

`ZenodoMetadata` does, and the file has none:

In [15]:
try:
    ZenodoMetadata.dataset(metadata).to_payload()
except ValidationError as error:
    print(error)
else:
    raise AssertionError("Zenodo should require a license")

license must be a non-empty string


Supply the license you have reviewed, as another override:

In [16]:
licensed = reviewed.with_overrides(license="cc-by-4.0")
zenodo_payload = ZenodoMetadata.dataset(licensed.to_publication_metadata()).to_payload()

assert zenodo_payload["metadata"]["license"] == "cc-by-4.0"
assert zenodo_payload["metadata"]["publication_date"] == "2026-04-16"

zenodo_payload

{'metadata': {'upload_type': 'dataset',
  'publication_date': '2026-04-16',
  'title': 'TTO/PMDco 3.0 A-Box mapping example of S355 steel tensile test data',
  'creators': [{'name': 'Jane Doe'}],
  'description': 'Data from 10 tensile tests. Material: S355. Test standard: DIN EN ISO 6892-1.',
  'access_right': 'open',
  'license': 'cc-by-4.0',
  'keywords': ['S355', 'tensile test'],
  'language': 'eng'}}

`access_right` and `language` in this payload are defaults of the Zenodo adapter, not facts from the RDF. Without an accepted date, the adapter would also fill in today's date as `publication_date`.

## The same review on PMDco 2 data

The second reference file uses PMDco 2 terms. Its CSVW tables and columns also have titles, but these structural resources don't qualify as publications, so automatic selection still finds a single candidate.

In [17]:
pmdco2 = harvest_publication_metadata_from_rdf(
    CORPUS / "pmdco2_tto_example.ttl"
).select_subject_automatically()

print(len(pmdco2.subjects), "subjects,", len(pmdco2.subject_candidates), "candidate")
print(pmdco2.subject.term)

560 subjects, 1 candidate
https://w3id.org/pmd/ao/tte/


The rules for this generation find the same facts through different terms, so the suggestions are equal:

In [18]:
pmdco2_description = pmdco2.description.suggestions[0]

assert pmdco2_description.value == description.value
assert pmdco2.keywords.suggestions[0].value == keywords.value

print([evidence.rule for evidence in pmdco2_description.evidence])

['pmdco2_tensile_test', 'pmdco2_material', 'pmdco2_test_standard']


This file does link creators, but none of them can be used as they are. Each is an ORCID of unknown kind; one has several conflicting names and the others have none. praeco doesn't guess: the creators stay visible and block conversion.

In [19]:
print(pmdco2.creators.status)
for creator in pmdco2.creators.observations:
    print(creator.term, f"kind={creator.kind}", f"names={len(creator.name_candidates)}")

try:
    pmdco2.accept_suggestion(pmdco2_description).to_publication_metadata()
except IncompleteHarvestError as error:
    print(len(error.diagnostics), "blocking diagnostics")
else:
    raise AssertionError("unresolved creators should block conversion")

unresolved
https://orcid.org/0000-0001-7192-7143 kind=unknown names=0
https://orcid.org/0000-0002-3717-7104 kind=unknown names=4
https://orcid.org/0000-0002-7094-5371 kind=unknown names=0
https://orcid.org/0000-0003-4971-3645 kind=unknown names=0
4 blocking diagnostics


A reviewed creator list resolves them, and the observations stay available for inspection:

In [20]:
pmdco2_reviewed = pmdco2.accept_suggestion(pmdco2_description).with_overrides(
    creators=(Person(name="Jane Doe"),)
)

print(pmdco2_reviewed.to_publication_metadata().title)
print(len(pmdco2_reviewed.creators.observations), "creator observations retained")

Tensile Test Ontology (TTO) data mapping example
4 creator observations retained


## Alternatives and exclusions

Neither reference file has competing values for a field, so this section uses a small inline example with two titles and two publication dates. Without a subject, the harvest offers its resources for an explicit choice:

In [21]:
turtle = """
@prefix dcat: <http://www.w3.org/ns/dcat#> .
@prefix dct: <http://purl.org/dc/terms/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .

<https://example.org/dataset> a dcat:Dataset ;
    dct:title "Tensile measurements", "Tensile test data" ;
    dct:description "Measurements from tensile tests." ;
    dct:creator <https://example.org/people/jane> ;
    dct:issued "2020-01-01", "2021-01-01" .

<https://example.org/people/jane> a foaf:Person ; foaf:name "Jane Doe" .
"""

example = harvest_publication_metadata_from_rdf(turtle)
for subject in example.subjects:
    print(subject.term)

https://example.org/dataset
https://example.org/people/jane


`select_subject` takes one of these records or an IRI. Conflicting values stay unresolved and block conversion, with all candidates kept:

In [22]:
conflicted = example.select_subject("https://example.org/dataset")

print(conflicted.title.status, [item.value for item in conflicted.title.candidates])
print(
    conflicted.publication_date.status,
    [item.value for item in conflicted.publication_date.candidates],
)

try:
    conflicted.to_publication_metadata()
except IncompleteHarvestError as error:
    print(error)
else:
    raise AssertionError("conflicting values should block conversion")

unresolved ['Tensile measurements', 'Tensile test data']
unresolved [datetime.date(2020, 1, 1), datetime.date(2021, 1, 1)]
title requires review.; publication_date requires review.


`select_candidate` chooses an observed value. `clear` deliberately leaves out an optional field, which also resolves its conflict. The candidates remain inspectable either way.

In [23]:
resolved = conflicted.select_candidate(conflicted.title.candidates[0]).clear(
    "publication_date"
)
resolved_metadata = resolved.to_publication_metadata()

assert resolved_metadata.title == "Tensile measurements"
assert resolved_metadata.publication_date is None
assert len(resolved.title.candidates) == 2
assert len(resolved.publication_date.candidates) == 2

print(resolved_metadata.title, resolved_metadata.publication_date)
print(resolved.title.origin, resolved.publication_date.origin)

Tensile measurements None
selection exclusion


## Next steps

To create a real dataset or deposition, pass the adapter objects to the service clients: `client.datasets.create(...)` in the [DataportalClient notebook](DataportalClient.ipynb) and `client.depositions.set_metadata(...)` in the [ZenodoClient notebook](ZenodoClient.ipynb).